# Step 6 — UK Gilt-Curve PCA (Level / Slope / Curvature)

Covariance-matrix PCA on the UK curve (2y/5y/10y/30y) and projection of the May-2026 move onto the level/slope/curvature components.

**Script notes**

```
Step 6 - UK gilt-curve PCA (level / slope / curvature)
Single clean source: BoE zero-coupon spot, maturities 2y 5y 10y 30y.
Convention: COVARIANCE matrix of daily CHANGES (bp) - preserves the fact that
different maturities have different volatilities, which is what makes PC1/2/3
interpretable as level / slope / curvature.

Test: does the May-2026 move project onto SLOPE/CURVATURE (long-end steepening,
the fiscal-premium signature) rather than pure LEVEL (uniform inflation shift)?
```

> Paths assume this notebook sits in a folder with a `./data/` subfolder holding the raw downloads, and that earlier steps have written their intermediate CSVs into the same folder. Run `load_yields` first.

In [1]:
import numpy as np
import pandas as pd

In [2]:
ch = pd.read_csv("./changes_bp.csv", index_col=0, parse_dates=True)
MAT = ["UK_2y", "UK_5y", "UK_10y", "UK_30y"]
mats = [2, 5, 10, 30]
X = ch[MAT]

In [3]:
# covariance-matrix PCA on daily changes (bp)
S = np.cov(X.values, rowvar=False)
ev, evec = np.linalg.eigh(S)
o = np.argsort(ev)[::-1]; ev, evec = ev[o], evec[:, o]
var_explained = ev/ev.sum()

In [4]:
# sign conventions for interpretability:
# PC1 (level): all-positive
if (evec[:,0] < 0).sum() > 2: evec[:,0] *= -1
# PC2 (slope): short-end negative, long-end positive (steepening = +)
if evec[0,1] > evec[-1,1]: evec[:,1] *= -1
# PC3 (curvature): belly vs wings - make wings positive
if (evec[0,2] + evec[-1,2]) < 0: evec[:,2] *= -1

In [5]:
EV = pd.DataFrame(evec[:, :3], index=[f"{m}y" for m in mats],
                  columns=["PC1_level", "PC2_slope", "PC3_curv"])

In [6]:
print("="*60)
print("UK GILT-CURVE PCA  (covariance of daily changes, bp)")
print(f"sample {X.index.min().date()} -> {X.index.max().date()}  n={len(X)}")
print("="*60)
print("\nVariance explained:")
for i in range(4):
    print(f"  PC{i+1}: {var_explained[i]*100:5.2f}%  (cum {var_explained[:i+1].sum()*100:5.2f}%)")
print("\nEigenvector loadings (shape of each component across the curve):")
print(EV.round(3).to_string())
print("\n  -> PC1 flat & same sign  = LEVEL")
print("     PC2 rises short->long  = SLOPE (steepening)")
print("     PC3 wings vs belly     = CURVATURE")

UK GILT-CURVE PCA  (covariance of daily changes, bp)
sample 2024-01-05 -> 2026-05-29  n=519

Variance explained:
  PC1: 90.11%  (cum 90.11%)
  PC2:  9.01%  (cum 99.12%)
  PC3:  0.75%  (cum 99.87%)
  PC4:  0.13%  (cum 100.00%)

Eigenvector loadings (shape of each component across the curve):
     PC1_level  PC2_slope  PC3_curv
2y       0.464     -0.633     0.548
5y       0.520     -0.269    -0.371
10y      0.531      0.179    -0.563
30y      0.482      0.703     0.495

  -> PC1 flat & same sign  = LEVEL
     PC2 rises short->long  = SLOPE (steepening)
     PC3 wings vs belly     = CURVATURE


In [7]:
# project observation scores
scores = (X.values - X.values.mean(0)) @ evec[:, :3]
sc = pd.DataFrame(scores, index=X.index, columns=["level", "slope", "curv"])

In [8]:
# --- characterise the May-2026 move ---
# cumulative curve change over the political episode window
def cum_move(a, b):
    seg = X.loc[a:b]
    return seg.sum()  # sum of daily changes = total change over window (bp)

In [9]:
win = ("2026-05-01", "2026-05-15")  # run-up to / incl. Big Read & worst day
move = cum_move(*win)
print(f"\nTotal UK curve change over {win[0]}..{win[1]} (bp):")
print(move.round(1).to_string())


Total UK curve change over 2026-05-01..2026-05-15 (bp):
UK_2y     10.0
UK_5y     16.1
UK_10y    15.8
UK_30y    13.6


In [10]:
# decompose that total move onto the 3 components
total_vec = move.values
proj = evec[:, :3].T @ total_vec   # coordinates on level/slope/curv
recon = evec[:, :3] @ proj
print("\nProjection of the May move onto components (bp of curve motion):")
for nm, p in zip(["level","slope","curv"], proj):
    print(f"   {nm:6s}: {p:6.1f}")
# share of move 'energy' explained by each
sq = proj**2
print("\nShare of May-move variance by component:")
for nm, s in zip(["level","slope","curv"], sq/sq.sum()):
    print(f"   {nm:6s}: {s*100:4.0f}%")


Projection of the May move onto components (bp of curve motion):
   level :   27.9
   slope :    1.8
   curv  :   -2.7

Share of May-move variance by component:
   level :   99%
   slope :    0%
   curv  :    1%


In [11]:
# compare: average daily move decomposition over whole sample (baseline mix)
print("\nFor reference, full-sample average |score| by component (typical daily mix):")
print((sc.abs().mean()).round(2).to_string())


For reference, full-sample average |score| by component (typical daily mix):
level    7.90
slope    2.34
curv     0.71


In [12]:
# slope score around the episode vs its own history (percentile)
slope_may = sc.loc["2026-05-01":"2026-05-15", "slope"]
pct = (sc["slope"] < slope_may.max()).mean()*100 if len(slope_may) else np.nan
print(f"\nPeak daily SLOPE score in early-May 2026 sits at {pct:.0f}th percentile of all days")


Peak daily SLOPE score in early-May 2026 sits at 94th percentile of all days


In [13]:
EV.to_csv("./ukcurve_loadings.csv")
sc.to_csv("./ukcurve_scores.csv")
pd.Series(var_explained, index=[f"PC{i+1}" for i in range(4)]).to_csv("./ukcurve_varexplained.csv")
print("\nsaved ukcurve_loadings.csv, ukcurve_scores.csv, ukcurve_varexplained.csv")


saved ukcurve_loadings.csv, ukcurve_scores.csv, ukcurve_varexplained.csv
